# Ingest drivers.json file
1. Read the file using spark dataframe reader API
1. Define and enforce schema (preserve the nested structure)
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table

In [0]:
dbutils.widgets.text("p_batch_id", "")

In [0]:
v_batch_id = dbutils.widgets.get("p_batch_id")
print(v_batch_id)

In [0]:
%run  ../00-common/01_Environmnet_config

In [0]:
%run  ../00-common/02_bronze_helpers

In [0]:
source_File = f"{landing_folder_path}/{v_batch_id}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"
print(source_File)
print(table_name)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType


name_schema = StructType(
    [
        StructField("givenName", StringType(), True),
        StructField("familyName", StringType(), True),
    ]
)

drivers_schema = StructType(
    [
        StructField("driverId", StringType(), True),
        StructField("name", name_schema, True),
        StructField("dateOfBirth", DateType(), True),
        StructField("nationality", StringType(), True),
        StructField("url", StringType(), True),
    ]
)

In [0]:
drivers_df = (
    spark.read.format("json")
    .schema(drivers_schema)
    .option("mode", "FAILFAST")
    .load(source_File)
)

In [0]:
constructor_final_df = add_ingestion_medatat(drivers_df)

In [0]:
write_to_bronze(constructor_final_df, table_name, v_batch_id)

In [0]:
%sql
select * from formula1_incr.bronze.drivers
where batch_id = '2025-01'